# 🧩 07 - Grouping, Summarizing, and Combining Data

So far, every question has been about the **whole** dataset or **one row** at a time. Real questions are usually about **groups**: *What's the survival rate in each class? Which state reports the most UFOs?* This notebook also shows how to stack tables and join related tables together.

## ✅ Learning Goals
- Count values with `value_counts()`, including as percentages and count-and-percent tables
- Summarize groups with `groupby()`, by one column or several, and know `.size()` from `.count()`
- Compute several summaries at once with named `.agg()`, and filter the groups afterwards
- Stack tables with `pd.concat()`, and keep track of where the rows came from
- Join related tables with `.merge()`: left vs. inner joins, `suffixes=`, `left_on`/`right_on`
- Check a join with its shape and `validate=`, so it can't silently duplicate rows

> **The core habit:** count categories with `value_counts()`, summarize groups with `groupby()`, stack similar tables with `concat()`, and combine related tables with `merge()`.

## 1. Setup

In [ ]:
import pandas as pd

titanic = pd.read_csv('titanic.csv')
titanic.head(3)

## 2. Counting Values
`value_counts()` counts how often each value appears, most common first. `normalize=True` turns the counts into **proportions**:

In [ ]:
titanic['Pclass'].value_counts()

In [ ]:
titanic['Pclass'].value_counts(normalize=True).round(3)   # share of passengers in each class

For a report, turn proportions into **percentages** with `.mul(100)`. `value_counts()` sorts by frequency, and `.sort_index()` sorts by the values themselves instead (1st, 2nd, 3rd class). Putting counts and percentages side by side makes a readable summary table:

In [ ]:
class_counts = titanic['Pclass'].value_counts().sort_index()
class_percent = titanic['Pclass'].value_counts(normalize=True).sort_index().mul(100).round(1)

pd.DataFrame({'passengers': class_counts, 'percent': class_percent})

Recap from notebook 02: `.unique()` lists the distinct values, `.nunique()` counts them, and `value_counts(dropna=False)` includes missing values in the count.

## 3. Group By: Split → Apply → Combine
`groupby()` does three things in one line:

```text
split    the rows into groups        (one group per class: 1, 2, 3)
apply    a summary to each group     (the mean of Survived)
combine  the answers into a result   (one row per group)
```

Since `Survived` is `1` or `0`, its **mean** is the survival rate:

In [ ]:
titanic.groupby('Pclass')['Survived'].mean()

Read it as: *"group by class, take the `Survived` column, and average it."* The same pattern works with any column and any summary: `.sum()`, `.count()`, `.min()`, `.max()`, `.median()`.

In [ ]:
titanic.groupby('Sex')['Fare'].median()

### `.size()` vs. `.count()`
`.size()` counts the **rows** in each group. `.count()` counts the **non-missing values** in one column. They differ whenever that column has gaps:

In [ ]:
pd.DataFrame({
    'rows (.size)':          titanic.groupby('Pclass').size(),
    'known ages (.count)':   titanic.groupby('Pclass')['Age'].count(),
})

### Grouping by More Than One Column
Pass a **list** to get one group per combination:

In [ ]:
titanic.groupby(['Pclass', 'Sex'])['Survived'].mean().round(2)

The result has a two-level index. `.reset_index()` turns the group labels back into ordinary columns, which gives you a regular DataFrame that's easier to sort, filter, or save:

In [ ]:
survival = titanic.groupby(['Pclass', 'Sex'])['Survived'].mean().round(2).reset_index()
survival.sort_values('Survived', ascending=False)

💡 `groupby(..., as_index=False)` gives you regular columns straight away, without needing `.reset_index()`. It's handy for reports:

In [ ]:
titanic.groupby('Pclass', as_index=False)['Fare'].mean().round(2)

## 4. Several Summaries at Once with `.agg()`
`.agg()` computes several summaries in one go. Name each output column with `new_name='function'`:

In [ ]:
titanic.groupby('Pclass')['Fare'].agg(
    passengers='count',
    average_fare='mean',
    highest_fare='max',
).round(2)

### Summaries of Different Columns
To summarize **different** columns in one table, group the whole DataFrame and give each output column a `(column, function)` pair:

```python
new_column_name=('existing_column', 'function')
```

In [ ]:
class_report = titanic.groupby('Pclass', as_index=False).agg(
    passengers=('PassengerId', 'count'),
    survival_rate=('Survived', 'mean'),
    average_age=('Age', 'mean'),
    average_fare=('Fare', 'mean'),
).round(2)
class_report

### Row-Level Values First, Then Group
To summarize something that isn't a column yet, **create it first**, then group. Here, family size = siblings/spouses + parents/children + the passenger:

In [ ]:
titanic['family_size'] = titanic['SibSp'] + titanic['Parch'] + 1

by_family = titanic.groupby('family_size', as_index=False).agg(
    passengers=('PassengerId', 'count'),
    survival_rate=('Survived', 'mean'),
).round(2)
by_family

### Filtering Groups After Summarizing
Some of those groups are tiny. Six passengers in families of 8 isn't enough to say much. To keep only groups that meet a condition, **summarize first, then filter the summary** like any DataFrame. (In SQL this is `GROUP BY` followed by `HAVING`.)

In [ ]:
by_family[by_family['passengers'] >= 20]

Traveling alone (30%) or in a large family was risky. Families of 2–4 did best.

## 5. Stacking Tables with `pd.concat()`
When two tables have the **same columns**, like this month's data and last month's, `pd.concat()` stacks them into one. `ignore_index=True` renumbers the rows `0, 1, 2, ...`, so labels aren't repeated:

In [ ]:
january = pd.DataFrame({'month': ['Jan', 'Jan'], 'store': ['North', 'South'], 'revenue': [12000, 9500]})
february = pd.DataFrame({'month': ['Feb', 'Feb'], 'store': ['North', 'South'], 'revenue': [14500, 9900]})

sales = pd.concat([january, february], ignore_index=True)
sales

In [ ]:
sales.groupby('store')['revenue'].sum()

💡 **Keep track of where rows came from.** These tables already had a `month` column. If yours don't, add one **before** stacking (`january['month'] = 'Jan'`), or you can't tell the rows apart afterwards.

### `concat` or `merge`?
```text
pd.concat()   stacks MORE ROWS of the same kind of data
              January rows + February rows  →  one longer table

.merge()      adds RELATED COLUMNS through a shared key
              purchases + buyer details     →  wider purchase rows
```

Before combining tables, ask:
- Am I adding **more rows of the same kind** of data? Use `concat`.
- Am I adding **related columns** through a shared key? Use `merge`.
- Which rows must **survive** if there's no match? Choose `'inner'` or `'left'` deliberately.

## 6. Joining Tables with `.merge()`
Real data is often split across tables that share a **key** column. Here, an order table only knows each product's ID, and the price lives in a separate table. `.merge()` lines the rows up by key, much like a SQL `JOIN`:

In [ ]:
order_lines = pd.DataFrame({
    'product_id': [101, 102, 101, 105],
    'quantity':   [2, 1, 3, 4],
})
prices = pd.DataFrame({
    'product_id': [101, 102, 103],
    'name':       ['Notebook', 'Backpack', 'Pen'],
    'price':      [4.99, 39.99, 1.50],
})

order_lines.merge(prices, on='product_id', how='left')

Product `105` isn't in the price table, so its `name` and `price` are `NaN`. That's the difference between the two most common joins:

| `how=` | Keeps... | Unmatched rows |
|---|---|---|
| `'left'` | **every** row of the left table | filled with `NaN` |
| `'inner'` | only rows whose key is in **both** tables | dropped |

In [ ]:
order_lines.merge(prices, on='product_id', how='inner')

### A Mini Database: Purchases and Buyers
Real data usually comes as several related tables. Here, each purchase records *who* bought it (`buyer_id`) and where it **shipped** (`state`). Buyer details, including where the buyer **lives** (also called `state`), are in a second table:

In [ ]:
purchases = pd.DataFrame({
    'purchase_id': [1001, 1002, 1003, 1004, 1005, 1006],
    'buyer_id':    [1, 2, 1, 3, 4, 2],
    'product_id':  [101, 102, 103, 101, 104, 102],
    'state':       ['IL', 'WI', 'IL', 'MN', 'IL', 'IN'],
    'amount':      [4.99, 39.99, 1.50, 4.99, 24.99, 39.99],
})
buyers = pd.DataFrame({
    'buyer_id':   [1, 2, 3, 4, 5],
    'buyer_name': ['Avery', 'Jordan', 'Riley', 'Morgan', 'Casey'],
    'state':      ['IL', 'WI', 'MN', 'IL', 'IL'],
})

#### Same Column Name in Both Tables: `suffixes=`
Both tables have a `state` column, so pandas can't keep both under one name. By default, it renames them `state_x` and `state_y`, which tells you nothing about which is which. Choose meaningful suffixes:

In [ ]:
print(purchases.merge(buyers, on='buyer_id').columns.tolist())

with_buyers = purchases.merge(buyers, on='buyer_id', how='left', suffixes=('_shipped', '_home'))
with_buyers[['purchase_id', 'buyer_name', 'state_shipped', 'state_home', 'amount']]

Jordan's second order shipped to Indiana, even though Jordan lives in Wisconsin. Without clear suffixes, that would be easy to misread.

#### Keys with Different Names: `left_on` / `right_on`
When the key columns have different names, name each side. Both key columns are kept, so drop the extra one once you've checked the result:

In [ ]:
orders = pd.DataFrame({'customer_number': [1, 2, 3], 'total': [19.99, 42.50, 10.00]})

orders.merge(buyers, left_on='customer_number', right_on='buyer_id', how='left').drop(columns=['buyer_id'])

#### Check Your Joins: Shape and `validate=`
A merge can quietly **duplicate** rows. If a key appears twice in the right table, every matching row on the left gets copied once for each. Suppose a buyer was accidentally entered twice:

In [ ]:
buyers_with_dupe = pd.concat([buyers, pd.DataFrame({'buyer_id': [2], 'buyer_name': ['Jordan B.'], 'state': ['WI']})],
                             ignore_index=True)

merged = purchases.merge(buyers_with_dupe, on='buyer_id', how='left')
print('purchases:', len(purchases), 'rows → after merge:', len(merged), 'rows')

Two purchases appeared out of nowhere, and every total computed from `merged` is now wrong. **Always compare row counts before and after a merge.** Better still, tell pandas what relationship you expect with `validate=`, so it stops with an error instead of silently duplicating:

In [ ]:
try:
    purchases.merge(buyers_with_dupe, on='buyer_id', how='left', validate='many_to_one')
except pd.errors.MergeError as error:
    print('MergeError:', str(error).splitlines()[0])

| `validate=` | Means |
|---|---|
| `'many_to_one'` | Left keys can repeat (many purchases per buyer), but each right key must be **unique** (one row per buyer) |
| `'one_to_many'` | The reverse |
| `'one_to_one'` | Each key appears once in **both** tables |

For joining details (buyers, products) onto records (purchases), `'many_to_one'` is almost always right. (We catch the error with `try`/`except` here so the notebook keeps running. In your own work, let it stop you!)

### A Real Join: UFO Sightings per Million People
`ufo.csv` has one row per sighting, and `state_abbr.json` maps each state's abbreviation to its population. Count the sightings per US state, then join the populations on:

In [ ]:
ufo = pd.read_csv('ufo.csv', usecols=['state', 'country'])
us = ufo[ufo['country'] == 'us']
sightings = us['state'].value_counts().rename_axis('state').reset_index(name='sightings')

# The JSON file is one object of {state: population}, so read it as a Series
population = pd.read_json('state_abbr.json', typ='series').rename('population').rename_axis('state').reset_index()

print(sightings.head(3))
print(population.head(3))

In [ ]:
joined = sightings.merge(population, on='state', how='left', validate='one_to_one')
print('States with no population:', joined['population'].isna().sum(), 'of', len(joined))

**Every** state failed to match! Look at the keys above: the sightings use `'ca'`, but the populations use `'CA'`. Keys must match **exactly**, so fix one side first:

In [ ]:
sightings['state'] = sightings['state'].str.upper()
joined = sightings.merge(population, on='state', how='left', validate='one_to_one')
print('Still unmatched:', joined[joined['population'].isna()]['state'].tolist())

joined['per_million'] = (joined['sightings'] / joined['population'] * 1_000_000).round(1)
joined.sort_values('per_million', ascending=False).head(5)

Only `PR` (Puerto Rico) is left unmatched, because it isn't in the population file. Dividing by population changes the story: California has the most sightings in total, but it ranks only 17th **per person**.

## 7. A Complete Walkthrough
**Question:** *For each shipping state and product department, how many purchases were there, what was the total revenue, and what was the average purchase?*

The purchases don't record the department, since that lives in a product table. So: **merge** in the detail you need (checking the join), **then** group and summarize:

In [ ]:
products = pd.DataFrame({
    'product_id':   [101, 102, 103, 104, 105],
    'product_name': ['Notebook', 'Backpack', 'Pen', 'Desk Lamp', 'Water Bottle'],
    'department':   ['Office', 'Travel', 'Office', 'Home', 'Travel'],
})

details = purchases.merge(products[['product_id', 'department']],    # only the columns you need
                          on='product_id', how='left', validate='many_to_one')
print('rows before/after:', len(purchases), len(details))

summary = (details
           .groupby(['state', 'department'], as_index=False)
           .agg(purchases=('purchase_id', 'count'),
                total_revenue=('amount', 'sum'),
                average_purchase=('amount', 'mean'))
           .round(2)
           .sort_values(['state', 'total_revenue'], ascending=[True, False]))
summary

Selecting only `product_id` and `department` before merging keeps the result tidy, and avoids clashing column names.

## 🏋️ Practice

### Practice 1 — Where Did They Board? (easy)
Use `value_counts()` on Titanic's `Embarked` column to show the **percentage** of passengers who boarded at each port, including the missing ones.

In [ ]:
# Your code here


### Practice 2 — Summarize by Port (medium)
Group Titanic by `Embarked` and use `.agg()` to show, for each port: the number of passengers, the average `Fare`, and the survival rate (the mean of `Survived`). Which port had the highest survival rate?

In [ ]:
# Your code here


### Practice 3 — Join, Then Group (harder)
Build these two tables, merge them on `student_id`, and then find the **average score for each major**. Which join (`'left'` or `'inner'`) did you use, and what happened to student `5`?

In [ ]:
scores = pd.DataFrame({'student_id': [1, 2, 3, 4, 5], 'score': [88, 92, 75, 81, 95]})
majors = pd.DataFrame({'student_id': [1, 2, 3, 4], 'major': ['CS', 'Math', 'CS', 'Biology']})
# Your code here


### Practice 4 — Only the Big Groups (medium)
Group Titanic by **both** `Embarked` and `Pclass`. Use `.agg()` with `(column, function)` pairs to get the number of passengers and the survival rate for each combination. Then keep only the groups with **at least 50** passengers, sorted by survival rate from highest to lowest.

In [ ]:
# Your code here


### Practice 5 — Join Carefully (harder)
Using `purchases` and `buyers` from section 6:
1. Merge them with `how='inner'`, `suffixes=('_shipped', '_home')`, and `validate='many_to_one'`, and compare the row count to `purchases`.
2. Which buyer never appears in the result, and why? Would a **left** join from `buyers` to `purchases` show them?
3. Find each buyer's total spending, using their `buyer_name`.

In [ ]:
# Your code here


## ✏️ Your Turn
1. Turn this list of purchases into a DataFrame. Then use `groupby` to find each buyer's **total** and **number** of purchases.
2. Add a `month` column, then stack two new purchases from a second DataFrame (with their own `month`) using `pd.concat()`, and recompute the totals per buyer **and** per month.
3. Use `value_counts()` to find the most frequent buyer, and show each buyer's share of purchases as a **percentage**.
4. In a comment, explain why `pd.concat()` fits monthly files, while `.merge()` fits adding buyer details to purchases.

In [ ]:
purchases = pd.DataFrame([
    {'purchase_id': 1, 'buyer': 'Avery',  'amount': 25.00},
    {'purchase_id': 2, 'buyer': 'Jordan', 'amount': 42.50},
    {'purchase_id': 3, 'buyer': 'Avery',  'amount': 18.75},
    {'purchase_id': 4, 'buyer': 'Sam',    'amount': 9.99},
])
# Your turn here


## 🔥 Challenge (Optional) — The Peak Year for UFO Sightings
Load the `date_time` column of `ufo.csv`, convert it with `pd.to_datetime(..., errors='coerce')`, and pull out the year with `.dt.year`. Then use `value_counts()` (or `groupby`) to find the year with the most sightings. *Hint:* a few hundred dates fail to convert. Why might that be? Look at some of the original text for the rows that became `NaT`.

In [ ]:
# Your investigation here


---
### 📝 Summary
| Task | Code |
|---|---|
| Count values | `df['col'].value_counts()`, add `normalize=True` for proportions, `.mul(100)` for percent |
| Summarize by group | `df.groupby('col')['other'].mean()` |
| Rows vs. values per group | `.size()` counts rows; `.count()` counts non-missing values |
| Group by several columns | `df.groupby(['a', 'b'], as_index=False)['other'].sum()` |
| Several summaries | `df.groupby('col').agg(total=('amount', 'sum'), average=('amount', 'mean'))` |
| Filter groups | Summarize first, then `summary[summary['count'] >= n]` |
| Stack tables | `pd.concat([df1, df2], ignore_index=True)` |
| Join tables | `left.merge(right, on='key', how='left')` |
| Same column names | `suffixes=('_left', '_right')` |
| Different key names | `left_on='a', right_on='b'` |
| Check a join | Compare `len()` before and after, and use `validate='many_to_one'` |

Next up: the **Pandas Wrangle Lab**, which puts the whole module together.